In [2]:
"""
Strategies:
1. Beam + Buddy + Refine (original)
2. Multi-Width + SA  
3. Block Merging Only
4. Buddy Chain Building
5. Random Restarts with Best Selection
"""

import cv2
import numpy as np
import os
import random
import math

GRID_SIZE = 4
NUM_PIECES = 16

BASE ="./results"
GT_DIR ="../data/correct"
COLOR_DIR = os.path.join(BASE, "enhanced_images_sliced", "puzzle_4x4")
OUTPUT_DIR = os.path.join(BASE, "4x4_out")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def sort_key(f):
    name = f.replace('piece_', '').split('.')[0]
    return int(name) if name.isdigit() else name

def load_pieces(folder):
    if not os.path.exists(folder): return None
    files = sorted([f for f in os.listdir(folder) if f.endswith(('.png', '.jpg'))], key=sort_key)
    pieces = [cv2.imread(os.path.join(folder, f)) for f in files]
    pieces = [p for p in pieces if p is not None]
    return pieces if len(pieces) == NUM_PIECES else None

def get_edge(img, side, w=1):
    if side == 'top': return img[:w, :, :]
    if side == 'bottom': return img[-w:, :, :]
    if side == 'left': return img[:, :w, :]
    if side == 'right': return img[:, -w:, :]

def lab_ssd(p1, p2, s1, s2, width=1):
    e1 = cv2.cvtColor(get_edge(p1, s1, width), cv2.COLOR_BGR2LAB).astype(np.float32)
    e2 = cv2.cvtColor(get_edge(p2, s2, width), cv2.COLOR_BGR2LAB).astype(np.float32)
    if s1 in ('left','right'): e1 = np.transpose(e1, (1,0,2))
    if s2 in ('left','right'): e2 = np.transpose(e2, (1,0,2))
    if e1.shape != e2.shape: return 1e9
    return np.sum((e1 - e2) ** 2) / e1.size

class UltraMegaSolver:
    def __init__(self, pieces):
        self.pieces = pieces
        self.n = len(pieces)
        
        # Standard matrices
        
        self.rc = np.full((self.n, self.n), np.inf)
        self.bc = np.full((self.n, self.n), np.inf)
        
        # Multi-width matrices
        self.rc2 = np.full((self.n, self.n), np.inf)
        self.bc2 = np.full((self.n, self.n), np.inf)
        
        for i in range(self.n):
            for j in range(self.n):
                if i != j:
                    self.rc[i, j] = lab_ssd(pieces[i], pieces[j], 'right', 'left', 1)
                    self.bc[i, j] = lab_ssd(pieces[i], pieces[j], 'bottom', 'top', 1)
                    self.rc2[i, j] = lab_ssd(pieces[i], pieces[j], 'right', 'left', 2)
                    self.bc2[i, j] = lab_ssd(pieces[i], pieces[j], 'bottom', 'top', 2)
        
        # Combined multi-width
        # Weighted sum of single-width and double-width compatibilities to get a more robust measure cus sometimes edges match better at different widths 
        self.rcm = 0.6 * self.rc + 0.4 * self.rc2
        self.bcm = 0.6 * self.bc + 0.4 * self.bc2
        
        # Find best buddies
        # argmin ==> returns index of minimum value along an axis
        
        self.hb, self.vb = {}, {}
        for a in range(self.n):
            b = np.argmin(self.rc[a, :])
            if np.argmin(self.rc[:, b]) == a: self.hb[a] = b
        for a in range(self.n):
            b = np.argmin(self.bc[a, :])
            if np.argmin(self.bc[:, b]) == a: self.vb[a] = b
    
    def evaluate(self, solution):
        grid = np.array(solution).reshape((4, 4))
        total = 0
        bb_count = 0
        loop_count = 0
        
        for r in range(4):
            for c in range(3):
                i, j = grid[r, c], grid[r, c+1]
                total += self.rc[i, j]
                if self.hb.get(i) == j: bb_count += 1
        
        for r in range(3):
            for c in range(4):
                i, j = grid[r, c], grid[r+1, c]
                total += self.bc[i, j]
                if self.vb.get(i) == j: bb_count += 1
        
        for r in range(3):
            for c in range(3):
                tl, tr = grid[r, c], grid[r, c+1]
                bl, br = grid[r+1, c], grid[r+1, c+1]
                loop = self.rc[tl, tr] + self.bc[tr, br] + self.rc[bl, br] + self.bc[tl, bl]
                if loop < 3000: loop_count += 1
        
        return total - bb_count * 3000 - loop_count * 2000
    
    def refine(self, sol, iterations=300):
        current = list(sol)
        best_c = self.evaluate(current)
        for _ in range(iterations):
            improved = False
            for i in range(16):
                for j in range(i+1, 16):
                    new = current[:]
                    new[i], new[j] = new[j], new[i]
                    nc = self.evaluate(new)
                    if nc < best_c:
                        best_c, current = nc, new
                        improved = True
                        break
                if improved: break
            if not improved: break
        return current
    
    def simulated_annealing(self, sol, iterations=1500):
        current = list(sol)
        current_cost = self.evaluate(current)
        best, best_cost = current[:], current_cost
        temp = 1500.0
        for _ in range(iterations):
            i1, i2 = random.sample(range(16), 2)
            new = current[:]
            new[i1], new[i2] = new[i2], new[i1]
            new_cost = self.evaluate(new)
            delta = new_cost - current_cost
            if delta < 0 or random.random() < math.exp(-delta / max(temp, 1)):
                current, current_cost = new, new_cost
                if current_cost < best_cost:
                    best, best_cost = current[:], current_cost
            temp *= 0.997
        return best
    
    # ============ STRATEGY 1: BEAM SEARCH ============
    def strategy_beam(self, use_multi=False):
        rc = self.rcm if use_multi else self.rc
        bc = self.bcm if use_multi else self.bc
        bb_mult = 0.05 if use_multi else 0.1
        
        best_sol, best_cost = None, float('inf')
        for start in range(16):
            beam = [(0.0, [start], {start})]
            for pos in range(1, 16):
                row, col = pos // 4, pos % 4
                cands = []
                for cost, pl, used in beam:
                    for p in range(16):
                        if p in used: continue
                        add = 0
                        if col > 0:
                            left = pl[pos-1]
                            add += rc[left, p]
                            if self.hb.get(left) == p: add *= bb_mult
                        if row > 0:
                            top = pl[pos-4]
                            add += bc[top, p]
                            if self.vb.get(top) == p: add *= bb_mult
                        cands.append((cost+add, pl+[p], used|{p}))
                cands.sort(key=lambda x: x[0])
                beam = cands[:500]
            if beam and beam[0][0] < best_cost:
                best_cost = beam[0][0]
                best_sol = beam[0][1]
        return best_sol
    
    # ============ STRATEGY 2: BUDDY GREEDY ============
    def strategy_buddy(self):
        best_sol, best_cost = None, float('inf')
        for start in list(self.hb.keys())[:8]:
            grid = [[-1]*4 for _ in range(4)]
            grid[0][0] = start
            if start in self.hb: grid[0][1] = self.hb[start]
            used = {x for row in grid for x in row if x >= 0}
            
            for pos in range(len(used), 16):
                row, col = pos // 4, pos % 4
                best_p, best_s = -1, float('inf')
                for p in range(16):
                    if p in used: continue
                    s = 0
                    if col > 0 and grid[row][col-1] >= 0:
                        s += self.rc[grid[row][col-1], p]
                        if self.hb.get(grid[row][col-1]) == p: s *= 0.1
                    if row > 0 and grid[row-1][col] >= 0:
                        s += self.bc[grid[row-1][col], p]
                        if self.vb.get(grid[row-1][col]) == p: s *= 0.1
                    if s < best_s: best_s, best_p = s, p
                if best_p >= 0:
                    grid[row][col] = best_p
                    used.add(best_p)
            
            sol = [grid[r][c] for r in range(4) for c in range(4)]
            if -1 not in sol:
                cost = self.evaluate(sol)
                if cost < best_cost: best_cost, best_sol = cost, sol
        return best_sol
    
    # ============ STRATEGY 3: BLOCK MERGING ============
    def strategy_blocks(self):
        class Block:
            def __init__(self, idx, img):
                self.idx = np.array([[idx]])
                self.img = img
        
        blocks = [Block(i, p) for i, p in enumerate(self.pieces)]
        
        def block_cost(b1, b2, side):
            lab1 = cv2.cvtColor(b1.img, cv2.COLOR_BGR2LAB).astype(np.float32)
            lab2 = cv2.cvtColor(b2.img, cv2.COLOR_BGR2LAB).astype(np.float32)
            if side == 'r':
                e1, e2 = lab1[:, -1, :], lab2[:, 0, :]
            else:
                e1, e2 = lab1[-1, :, :], lab2[0, :, :]
            return np.sum((e1 - e2) ** 2) / e1.size
        
        while len(blocks) > 1:
            best_merge, best_score = None, float('inf')
            for i in range(len(blocks)):
                for j in range(len(blocks)):
                    if i == j: continue
                    b1, b2 = blocks[i], blocks[j]
                    if b1.idx.shape[1] + b2.idx.shape[1] <= 4 and b1.idx.shape[0] == b2.idx.shape[0]:
                        s = block_cost(b1, b2, 'r')
                        if s < best_score: best_score, best_merge = s, (i, j, 'r')
                    if b1.idx.shape[0] + b2.idx.shape[0] <= 4 and b1.idx.shape[1] == b2.idx.shape[1]:
                        s = block_cost(b1, b2, 'b')
                        if s < best_score: best_score, best_merge = s, (i, j, 'b')
            
            if best_merge is None: break
            i, j, side = best_merge
            b1, b2 = blocks[i], blocks[j]
            
            if side == 'r':
                new_idx = np.hstack([b1.idx, b2.idx])
                new_img = np.hstack([b1.img, b2.img])
            else:
                new_idx = np.vstack([b1.idx, b2.idx])
                new_img = np.vstack([b1.img, b2.img])
            
            new_block = Block(-1, new_img)
            new_block.idx = new_idx
            
            for x in sorted([i, j], reverse=True): blocks.pop(x)
            blocks.append(new_block)
        
        if len(blocks) == 1 and blocks[0].idx.shape == (4, 4):
            return blocks[0].idx.flatten().tolist()
        return None
    
    # ============ STRATEGY 4: RANDOM RESTART ============
    def strategy_random_restart(self, n_restarts=20):
        best_sol, best_cost = None, float('inf')
        
        for _ in range(n_restarts):
            # Random permutation
            perm = list(range(16))
            random.shuffle(perm)
            
            # SA to improve
            refined = self.simulated_annealing(perm, 500)
            cost = self.evaluate(refined)
            
            if cost < best_cost:
                best_cost = cost
                best_sol = refined
        
        return best_sol
    
    # ============ STRATEGY 5: REVERSE ORDER BEAM ============
    def strategy_reverse_beam(self):
        """Build from bottom-right instead of top-left"""
        best_sol, best_cost = None, float('inf')
        
        for start in range(16):
            beam = [(0.0, [start], {start})]
            
            for pos in range(1, 16):
                # Fill from position 15 down to 0
                actual_pos = 15 - pos
                row, col = actual_pos // 4, actual_pos % 4
                
                cands = []
                for cost, pl, used in beam:
                    for p in range(16):
                        if p in used: continue
                        add = 0
                        # Check right neighbor (already placed)
                        if col < 3:
                            right_idx = actual_pos + 1
                            if right_idx < 16 and len(pl) > (15 - right_idx):
                                right = pl[15 - right_idx]
                                add += self.rc[p, right]
                                if self.hb.get(p) == right: add *= 0.1
                        # Check bottom neighbor
                        if row < 3:
                            bottom_idx = actual_pos + 4
                            if bottom_idx < 16 and len(pl) > (15 - bottom_idx):
                                bottom = pl[15 - bottom_idx]
                                add += self.bc[p, bottom]
                                if self.vb.get(p) == bottom: add *= 0.1
                        cands.append((cost+add, pl+[p], used|{p}))
                
                cands.sort(key=lambda x: x[0])
                beam = cands[:300]
            
            if beam:
                # Reverse to get correct order
                sol = beam[0][1][::-1]
                cost = self.evaluate(sol)
                if cost < best_cost:
                    best_cost = cost
                    best_sol = sol
        
        return best_sol
    
    def solve(self):
        candidates = []
        
        # Strategy 1a: Standard Beam
        sol = self.strategy_beam(use_multi=False)
        if sol and len(sol) == 16: candidates.append(sol)
        
        # Strategy 1b: Multi-width Beam
        sol = self.strategy_beam(use_multi=True)
        if sol and len(sol) == 16: candidates.append(sol)
        
        # Strategy 2: Buddy Greedy
        sol = self.strategy_buddy()
        if sol and len(sol) == 16: candidates.append(sol)
        
        # Strategy 3: Block Merging
        sol = self.strategy_blocks()
        if sol and len(sol) == 16: candidates.append(sol)
        
        # Strategy 4: Random Restart
        sol = self.strategy_random_restart()
        if sol and len(sol) == 16: candidates.append(sol)
        
        # Strategy 5: Reverse Beam
        sol = self.strategy_reverse_beam()
        if sol and len(sol) == 16: candidates.append(sol)
        
        if not candidates: return list(range(16))
        
        # Refine all candidates with both methods
        refined = []
        for c in candidates:
            # Deep refine
            r1 = self.refine(c)
            # SA
            r2 = self.simulated_annealing(r1)
            # Final refine
            r3 = self.refine(r2)
            refined.append((r3, self.evaluate(r3)))
        
        # Pick best
        refined.sort(key=lambda x: x[1])
        return refined[0][0]

def solve(pieces):
    return UltraMegaSolver(pieces).solve()

def assemble(pieces, order):
    h, w = pieces[0].shape[:2]
    result = np.zeros((4*h, 4*w, 3), dtype=np.uint8)
    for pos, idx in enumerate(order):
        r, c = pos // 4, pos % 4
        result[r*h:(r+1)*h, c*w:(c+1)*w] = pieces[idx]
    return result

def mse(a, b):
    if a.shape != b.shape: b = cv2.resize(b, (a.shape[1], a.shape[0]))
    return np.mean((a.astype(float) - b.astype(float)) ** 2)

def evaluate(limit=110):
    folders = sorted([f for f in os.listdir(COLOR_DIR) if os.path.isdir(os.path.join(COLOR_DIR, f))],
                     key=lambda x: int(x) if x.isdigit() else x)
    correct, total = 0, 0
    
    print("-"*60)
    print (f"{'Puzzle':<6} | {'MSE':<12} | {'Status'}")
    print("-"*60)
    
    for folder in folders[:limit]:
        pieces = load_pieces(os.path.join(COLOR_DIR, folder))
        if pieces is None: continue
        
        try:
            order = solve(pieces)
            result = assemble(pieces, order)
            
            # Save assembled image
            out_path = os.path.join(OUTPUT_DIR, f"{folder}_assembled.png")
            cv2.imwrite(out_path, result)
            
            gt_path = os.path.join(GT_DIR, f"{folder}.png")
            if not os.path.exists(gt_path): gt_path = os.path.join(GT_DIR, f"{folder}.jpg")
            
            error = mse(result, cv2.imread(gt_path))
            status = "PASS" if error < 1000 else "FAIL"
            if status == "PASS": correct += 1
        except Exception as e:
            status, error = "ERR", -1
        
        print(f"{folder:<6} | {error:<12.2f} | {status}")
        total += 1
    
    print(f"\nAccuracy: {correct}/{total} ({100*correct/total:.2f}%)")

if __name__ == "__main__":
    evaluate(110)


------------------------------------------------------------
Puzzle | MSE          | Status
------------------------------------------------------------
0      | 130.16       | PASS
1      | 32.69        | PASS
2      | 63.09        | PASS
3      | 44.77        | PASS
4      | 48.36        | PASS
5      | 65.50        | PASS
6      | 54.38        | PASS
7      | 33.92        | PASS
8      | 56.45        | PASS
9      | 62.20        | PASS
10     | 52.68        | PASS
11     | 63.37        | PASS
12     | 47.52        | PASS
13     | 77.19        | PASS
14     | 117.13       | PASS
15     | 123.04       | PASS
16     | 37.99        | PASS
17     | 51.30        | PASS
18     | 742.78       | PASS
19     | 62.46        | PASS
20     | 79.46        | PASS
21     | 24.20        | PASS
22     | 41.69        | PASS
23     | 60.20        | PASS
24     | 26.16        | PASS
25     | 2249.76      | FAIL
26     | 7895.27      | FAIL
27     | 33.45        | PASS
28     | 41.76        | PASS
29    